<a href="https://colab.research.google.com/github/OscarPasasin/Parcial4-PasasinOscar-1704862022/blob/main/notebooks/AsociacionClaveA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import warnings
#Ignorar advertencias
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [43]:
import pandas as pd
import numpy as np

In [31]:
url = "https://raw.githubusercontent.com/OscarPasasin/Parcial4-PasasinOscar-1704862022/refs/heads/main/Archivos/clave_A_asociacion.csv"
df = pd.read_csv(url)
df.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,A-T0001,A-C0058,2026-01-07,Limpieza,Detergente,1,App
1,A-T0001,A-C0058,2026-01-07,Panaderia,Pastelito,1,App
2,A-T0001,A-C0058,2026-01-07,Limpieza,Suavizante,1,App
3,A-T0002,A-C0044,2026-02-14,Panaderia,Pastelito,3,Web
4,A-T0003,A-C0002,2026-03-19,Lacteos,Queso,2,Web


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580 entries, 0 to 579
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  580 non-null    object
 1   cliente_id      580 non-null    object
 2   fecha           580 non-null    object
 3   categoria       580 non-null    object
 4   item            580 non-null    object
 5   cantidad        580 non-null    int64 
 6   canal           579 non-null    object
dtypes: int64(1), object(6)
memory usage: 31.8+ KB


In [33]:
df.isnull().sum()

,0
transaccion_id,0
cliente_id,0
fecha,0
categoria,0
item,0
cantidad,0
canal,1


In [34]:
df.duplicated().sum()

np.int64(1)

In [35]:
df = df.drop_duplicates()
df['canal'] = df['canal'].fillna('No Especificado')

In [23]:
# Agrupamos por 'transaccion_id' e 'item', y contamos la presencia.
basket = (df.groupby(['transaccion_id', 'item'])['cantidad'].sum().unstack().reset_index().fillna(0).set_index('transaccion_id'))

In [36]:
# Convertimos los valores mayores a 0 en True (1) y los iguales a 0 en False (0)
def encode_units(x):
    return x > 0

basket_sets = basket.applymap(encode_units)

print("Matriz transsaccional reparada")
display(basket_sets.head())

Matriz transsaccional reparada


/tmp/ipykernel_2542/3343340923.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_sets = basket.applymap(encode_units)


item,Agua,Arroz,Cafe,Cloro,Crema,Detergente,Frijoles,Galletas,Jabon,Jugo,Leche,Pan,Pasta,Pastelito,Queso,Salsa,Suavizante,Te,Tortilla,Yogurt
transaccion_id,,,,,,,,,,,,,,,,,,,,
A-T0001,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,False
A-T0002,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
A-T0003,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
A-T0004,True,True,False,False,False,False,False,True,False,False,False,False,True,False,True,True,False,False,False,False
A-T0005,False,False,True,False,False,False,False,False,False,False,True,True,False,False,True,False,False,False,False,False


In [37]:
from mlxtend.frequent_patterns import apriori, association_rules

#Definimos un soporte mínimo del 3% para capturar patrones comunes
frequent_itemsets = apriori(basket_sets, min_support=0.03, use_colnames=True)

#Generamos las reglas de asociación basándonos en la métrica Lift
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

In [39]:
# Ordenamos las reglas por Lift y seleccionamos las top 10
top_10_rules = rules.sort_values(by='lift', ascending=False).head(10)

In [41]:
# Limpiamos la visualización mostrando solo las métricas clave de negocio
display(top_10_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

,antecedents,consequents,support,confidence,lift
227,"(Frijoles, Cafe)","(Queso, Pan)",0.054545,0.750000,5.380435
226,"(Queso, Pan)","(Frijoles, Cafe)",0.054545,0.391304,5.380435
170,"(Pastelito, Detergente)",(Suavizante),0.036364,1.000000,5.156250
175,(Suavizante),"(Pastelito, Detergente)",0.036364,0.187500,5.156250
224,"(Frijoles, Queso)","(Cafe, Pan)",0.054545,0.750000,4.950000
229,"(Cafe, Pan)","(Frijoles, Queso)",0.054545,0.360000,4.950000
128,"(Frijoles, Cafe)",(Galletas),0.030303,0.416667,4.296875
133,(Galletas),"(Frijoles, Cafe)",0.030303,0.312500,4.296875
185,(Galletas),"(Frijoles, Queso)",0.030303,0.312500,4.296875
180,"(Frijoles, Queso)",(Galletas),0.030303,0.416667,4.296875
